# WEC Performance Analysis - Phase 1 | Absolute Variant

Unified pipeline for the Absolute Performance Assessment of 12 Wave Energy Converters (WECs/Buoys). This module consolidates model training, inference, anomaly flagging, visualisation, and terminal reporting.

## Pipeline stages
* **A** Data ingestion and feature engineering
* **B** Temporal train/test split (80 / 20)
* **C** XGBoost regression training + metric extraction
* **D** Full-dataset inference and residual computation
* **E** Absolute anomaly flagging (residual < -1.5 * RMSE_test)
* **F** Visualisation (3-panel, 2x2 GridSpec) saved to PNG
* **G** Asset Performance Report emitted to the logger (Epoch 3)

In [5]:
from __future__ import annotations

import logging
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=UserWarning)


# ---------------------------------------------------------------------------
# Global constants
# ---------------------------------------------------------------------------

DATA_PATH: str = "dataset2/wec_c5_mock_data_epochs.csv"
OUTPUT_PATH: str = "plots/phase1/wec_phase1_absolute.png"

PHASE1_CSV_OUT: str = "dataset2/wec_phase1_outputs.csv"
PHASE1_MODEL_OUT: str = "dataset2/wec_phase1_xgboost.joblib"

TIMESTAMP_COL: str = "PCTimeStamp"
TARGET_COL: str = "Energy_Generation_kW"
BUOY_ID_COL: str = "Buoy_ID"
EPOCH_COL: str = "Epoch_Marker"

# Expected buoy identifiers, ordered for consistent visualisation
BUOY_ORDER: List[str] = [f"Boia_{i}" for i in range(1, 13)]

# Model feature set
FEATURE_COLS: List[str] = [
    "Hs__m",
    "Te__s",
    "Wave_Power_Flux",
    "H1/3__m",
    "H1/10__m",
    "Hmax__m",
    "HTmax__m",
    "Havg__m",
    "Hsms__m",
    "NumberOfWaves",
    "THmax__s",
    "Tavg__s",
    "Tmax__s",
    "hour",
    "month",
]

# Temporal split fraction
TEST_FRACTION: float = 0.20
TRAIN_CUTOFF_DATE: str = "2025-05-01"

# Anomaly detection multiplier applied to RMSE_test
ANOMALY_SIGMA_FACTOR: float = 1.5

# Confidence band multiplier for the P10-P90 visual band
CONFIDENCE_Z: float = 1.28

# Physical generation bounds in kW
GEN_MIN_KW: float = 0.0
GEN_MAX_KW: float = 350.0

# XGBoost hyper-parameters
XGB_PARAMS: Dict = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}


HEALTHY_BUOYS: List[str] = [f"Boia_{i}" for i in range(1, 9)]
DEGRADED_BUOYS: List[str] = [f"Boia_{i}" for i in range(9, 13)]
ALL_BUOYS: List[str] = HEALTHY_BUOYS + DEGRADED_BUOYS

# Matplotlib / Seaborn aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("paper", font_scale=1.2)

## Core Functions
Definição das funções para engenharia de features, treino do XGBoost, inferência, visualização e geração de relatórios.

In [ ]:
# ---------------------------------------------------------------------------
# Stage A -- Data ingestion and feature engineering
# ---------------------------------------------------------------------------
def load_and_engineer_features(csv_path: str) -> pd.DataFrame:
    """Load raw sensor data from CSV, impute missing values, and compute derived / temporal features."""
    logger.info("Stage A -- Loading data from: %s", csv_path)
    df: pd.DataFrame = pd.read_csv(csv_path, parse_dates=[TIMESTAMP_COL])
    logger.info("Raw shape: %s", df.shape)

    df = df.sort_values(TIMESTAMP_COL).reset_index(drop=True)

    numeric_cols: List[str] = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        col_std: float = df[col].std()
        if col_std == 0:
            continue
        z: pd.Series = (df[col] - df[col].mean()).abs() / col_std
        n_flagged: int = int((z > 4.0).sum())
        if n_flagged:
            logger.debug("Column '%s': flagging %d outlier(s) as NaN", col, n_flagged)
            df.loc[z > 4.0, col] = np.nan

    missing_total: int = int(df[numeric_cols].isna().sum().sum())
    if missing_total:
        logger.info("Imputing %d missing value(s) via median strategy", missing_total)
        imputer = SimpleImputer(strategy="median")
        df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

    df["Wave_Power_Flux"] = 0.49 * (df["Hs__m"] ** 2) * df["Te__s"]
    logger.info("Engineered feature 'Wave_Power_Flux' computed (0.49 * Hs^2 * Te)")

    df["hour"] = df[TIMESTAMP_COL].dt.hour
    df["month"] = df[TIMESTAMP_COL].dt.month
    logger.info("Temporal features extracted: hour, month")

    logger.info("Preprocessed shape: %s", df.shape)
    return df

# ---------------------------------------------------------------------------
# Stage B -- Temporal split
# ---------------------------------------------------------------------------

def temporal_split(
    df: pd.DataFrame,
    test_fraction: float = TEST_FRACTION,
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Produce a chronological 80/20 train/test split without data leakage.

    Parameters
    ----------
    df : pd.DataFrame
        Preprocessed DataFrame sorted by timestamp.
    test_fraction : float
        Fraction of rows reserved for the test set (most recent portion).

    Returns
    -------
    X_train, y_train, X_test, y_test : tuple of DataFrames and Series
    """
    split_idx: int = int(len(df) * (1.0 - test_fraction))

    X: pd.DataFrame = df[FEATURE_COLS].copy()
    y: pd.Series = df[TARGET_COL].copy()
    buoys: pd.Series = df[BUOY_ID_COL].copy()

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    buoys_train, buoys_test = buoys.iloc[:split_idx], buoys.iloc[split_idx:]

    logger.info(
        "Stage B -- Temporal split: train=%d rows | test=%d rows (%.0f%% / %.0f%%)",
        len(X_train),
        len(X_test),
        (1.0 - test_fraction) * 100,
        test_fraction * 100,
    )
    return X_train, y_train, X_test, y_test, buoys_train, buoys_test


# ---------------------------------------------------------------------------
# Stage C -- Model training and metric extraction
# ---------------------------------------------------------------------------

def train_and_evaluate(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    buoys_test: pd.Series,
    params: Optional[Dict] = None,
) -> Tuple[XGBRegressor, float]:
    """
    Train an XGBoost regressor and return the fitted model together with the
    test-set RMSE extracted dynamically (never hardcoded).

    Parameters
    ----------
    X_train, y_train : training features and target.
    X_test, y_test   : held-out evaluation features and target.
    params           : optional XGBoost hyper-parameter dictionary.

    Returns
    -------
    model : XGBRegressor
        Fitted model ready for inference.
    rmse_test : float
        Root Mean Squared Error on the test partition.
    """
    effective_params: Dict = params or XGB_PARAMS
    model = XGBRegressor(**effective_params)

    logger.info(
        "Stage C -- Training XGBoost on %d samples, %d features...",
        len(X_train),
        X_train.shape[1],
    )
    model.fit(X_train, y_train)
    logger.info("Training complete.")

    y_pred_train: np.ndarray = model.predict(X_train)
    rmse_train: float = float(np.sqrt(mean_squared_error(y_train, y_pred_train)))
    mae_train: float = float(mean_absolute_error(y_train, y_pred_train))
    r2_train: float = float(r2_score(y_train, y_pred_train))
    
    y_pred_test: np.ndarray = model.predict(X_test)
    rmse_test_global: float = float(np.sqrt(mean_squared_error(y_test, y_pred_test)))
    mae_test_global: float = float(mean_absolute_error(y_test, y_pred_test))
    r2_test_global: float = float(r2_score(y_test, y_pred_test))

    mask_healthy = buoys_test.isin(HEALTHY_BUOYS)
    mask_degraded = buoys_test.isin(DEGRADED_BUOYS)

    r2_test_healthy = r2_score(y_test[mask_healthy], y_pred_test[mask_healthy])
    rmse_test_healthy = np.sqrt(mean_squared_error(y_test[mask_healthy], y_pred_test[mask_healthy]))
    mae_test_healthy = float(mean_absolute_error(y_test[mask_healthy], y_pred_test[mask_healthy]))

    r2_test_degraded = r2_score(y_test[mask_degraded], y_pred_test[mask_degraded])
    rmse_test_degraded = np.sqrt(mean_squared_error(y_test[mask_degraded], y_pred_test[mask_degraded]))
    mae_test_degraded = float(mean_absolute_error(y_test[mask_degraded], y_pred_test[mask_degraded]))

    logger.info("--- Baseline Metrics (In-Sample / Epoch 1 / Train set) ---")
    logger.info("  RMSE : %.4f kW", rmse_train)
    logger.info("  MAE  : %.4f kW", mae_train)
    logger.info("  R^2  : %.4f ", r2_train)
    
    logger.info("--- Global Test Set Metrics (Out-of-Sample / Test set) ---")
    logger.info("  RMSE : %.4f kW ", rmse_test_global)
    logger.info("  MAE  : %.4f kW", mae_test_global)
    logger.info("  R^2  : %.4f ", r2_test_global)
    logger.info("-------------------------------------------------")

    logger.info("--- Healthy Test Set Metrics (Out-of-Sample / Test set) ---")
    logger.info("  RMSE : %.4f kW ", rmse_test_healthy)
    logger.info("  MAE  : %.4f kW", mae_test_healthy)
    logger.info("  R^2  : %.4f ", r2_test_healthy)
    logger.info("-------------------------------------------------")

    logger.info("--- Degraded Test Set Metrics (Out-of-Sample / Test set) ---")
    logger.info("  RMSE : %.4f kW  [used dynamically for anomalies]", rmse_test_degraded)
    logger.info("  MAE  : %.4f kW", mae_test_degraded)
    logger.info("  R^2  : %.4f ", r2_test_degraded)
    logger.info("-------------------------------------------------")

    importances: pd.Series = (
        pd.Series(model.feature_importances_, index=X_train.columns)
        .sort_values(ascending=False)
    )
    logger.info("Top-5 feature importances:\n%s", importances.head(5).to_string())

    return model, rmse_train



# ---------------------------------------------------------------------------
# Stage D & E -- Full-dataset inference and anomaly flagging
# ---------------------------------------------------------------------------
def run_inference_and_flag(df: pd.DataFrame, model: XGBRegressor, rmse_test: float) -> pd.DataFrame:
    """Apply the trained model to 100% of the dataset, compute residuals, and set the absolute anomaly flag."""
    logger.info("Stage D/E -- Full inference on %d rows + anomaly flagging", len(df))

    predictions: np.ndarray = model.predict(df[FEATURE_COLS])
    df = df.copy()
    df["Predicted_Energy_kW"] = np.clip(predictions, GEN_MIN_KW, GEN_MAX_KW)

    df["Absolute_Residual"] = df[TARGET_COL] - df["Predicted_Energy_kW"]

    anomaly_threshold: float = -ANOMALY_SIGMA_FACTOR * rmse_test
    df["Is_Absolute_Anomaly"] = df["Absolute_Residual"] < anomaly_threshold

    n_anomalies: int = int(df["Is_Absolute_Anomaly"].sum())
    logger.info("Global anomaly rate: %d / %d timestamps", n_anomalies, len(df))
    return df
# ---------------------------------------------------------------------------
# Stage F -- Visualisation
# ---------------------------------------------------------------------------
def _build_panel_1_feature_importance(ax: plt.Axes, model: XGBRegressor, top_n: int = 5) -> None:
    importances: pd.Series = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    importances.tail(top_n).plot(kind="barh", color="#2c3e50", ax=ax)
    
    ax.set_title("A. Feature Importance\n(Phase 1 Output --> DEA Input)", fontweight="bold", fontsize=20)
    ax.set_xlabel("Importance Score (XGBoost)", fontsize=18)
    ax.tick_params(axis="both", labelsize=16)

def _build_panel_3_forecast(ax: plt.Axes, df: pd.DataFrame, rmse_test: float, buoy_id: str = "Boia_9", train_cutoff: str = TRAIN_CUTOFF_DATE) -> None:
    half_band: float = CONFIDENCE_Z * rmse_test
    cutoff_dt: pd.Timestamp = pd.to_datetime(train_cutoff)

    df_buoy: pd.DataFrame = df[df[BUOY_ID_COL] == buoy_id].set_index(TIMESTAMP_COL)[[TARGET_COL, "Predicted_Energy_kW"]].resample("D").mean()
    df_buoy["P10"] = (df_buoy["Predicted_Energy_kW"] - half_band).clip(lower=GEN_MIN_KW)
    df_buoy["P90"] = (df_buoy["Predicted_Energy_kW"] + half_band).clip(upper=GEN_MAX_KW)

    train_mask: pd.Series = df_buoy.index < cutoff_dt
    df_train: pd.DataFrame = df_buoy[train_mask]
    df_test: pd.DataFrame = df_buoy[~train_mask]

    ax.plot(df_buoy.index, df_buoy[TARGET_COL], label="Actual Generation", color="#e74c3c", linewidth=2)
    ax.plot(df_train.index, df_train["Predicted_Energy_kW"], label="Model (Train / In-Sample)", color="gray", linestyle="--", linewidth=1.5)
    ax.plot(df_test.index, df_test["Predicted_Energy_kW"], label="Forecast (Test / Out-of-Sample)", color="#27ae60", linestyle="--", linewidth=2.5)
    ax.fill_between(df_test.index, df_test["P10"], df_test["P90"], color="#2ecc71", alpha=0.3, label=f"Confidence Band (+/-{CONFIDENCE_Z} RMSE = +/-{half_band:.1f} kW)")

    ax.axvline(cutoff_dt, color="black", linestyle="-", lw=1.5, label="Train Cutoff (80%)")
    ax.axvline(pd.to_datetime("2025-05-15"), color="gray", linestyle=":", alpha=0.7)

    ax.set_title(f"C. Probabilistic Forecast: {buoy_id} (Actual vs Expected)", fontweight="bold", fontsize=20)
    ax.set_ylabel("Power (kW)", fontsize=18)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.tick_params(axis="both", labelsize=16)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha="right", fontsize=16)
    ax.legend(fontsize=16, loc="lower left")

def _build_panel_2_anomaly_bar(ax: plt.Axes, df: pd.DataFrame, epoch: int = 3) -> None:
    df_epoch: pd.DataFrame = df[df[EPOCH_COL] == epoch].copy()
    anomaly_pct: pd.Series = df_epoch.groupby(BUOY_ID_COL)["Is_Absolute_Anomaly"].mean().reindex(BUOY_ORDER).fillna(0.0) * 100.0

    anomaly_pct.index = anomaly_pct.index.str.replace("Boia_", "Buoy ")

    colors: List[str] = ["#c0392b" if buoy in [f"Buoy {i}" for i in range(9, 13)] else "#2c3e50" for buoy in anomaly_pct.index]
    bars = ax.bar(anomaly_pct.index, anomaly_pct.values, color=colors, edgecolor="white", linewidth=0.6)

    ax.axhline(10.0, color="orange", linestyle="--", linewidth=1.2, label="10% Reference Line")

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#2c3e50", label="Healthy Fleet (Buoys 1-8)"), 
        Patch(facecolor="#c0392b", label="Degraded Fleet (Buoys 9-12)")
    ]
    ax.legend(handles=legend_elements, fontsize=16, loc="upper left")
    
    ax.set_title(f"B. Absolute Anomaly Rate per Buoy - Epoch {epoch}\n(% Timestamps with Residual < -1.5 * RMSE_test)", fontweight="bold", fontsize=20)
    ax.set_xlabel("WEC Asset (Buoy ID)", fontsize=18)
    ax.set_ylabel("Anomaly Rate (%)", fontsize=18)
    ax.tick_params(axis="x", labelrotation=30, labelsize=16)
    ax.tick_params(axis="y", labelsize=16)
    ax.set_ylim(bottom=0)

def generate_figure(df: pd.DataFrame, model: XGBRegressor, rmse_test: float, output_path: str = OUTPUT_PATH) -> None:
    logger.info("Stage F -- Composing 3-panel figure")
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    fig = plt.figure(figsize=(18, 14))
    
    gs = fig.add_gridspec(nrows=2, ncols=2, hspace=0.45, wspace=0.22)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, :])


    _build_panel_1_feature_importance(ax1, model)
    _build_panel_2_anomaly_bar(ax2, df, epoch=3)
    _build_panel_3_forecast(ax3, df, rmse_test)

    plt.savefig(output_path, dpi=600, bbox_inches="tight")
    logger.info("Figure saved to: %s", Path(output_path).resolve())
    plt.close(fig)

# ---------------------------------------------------------------------------
# Stage G & H -- Reporting and Export
# ---------------------------------------------------------------------------
def emit_asset_performance_report(df: pd.DataFrame, epoch: int = 3) -> None:
    df_epoch: pd.DataFrame = df[df[EPOCH_COL] == epoch].copy()
    separator: str = "=" * 70

    logger.info(separator)
    logger.info("ASSET PERFORMANCE REPORT -- Epoch %d", epoch)
    logger.info("Anomaly definition: Residual < -%.1f * RMSE_test", ANOMALY_SIGMA_FACTOR)
    logger.info(separator)

    mask_healthy = df_epoch[BUOY_ID_COL].isin(HEALTHY_BUOYS)
    mask_degraded = df_epoch[BUOY_ID_COL].isin(DEGRADED_BUOYS)
    
    y_true_h = df_epoch[mask_healthy][TARGET_COL]
    y_pred_h = df_epoch[mask_healthy]["Predicted_Energy_kW"]
    r2_healthy = r2_score(y_true_h, y_pred_h) if not y_true_h.empty else np.nan
    
    y_true_d = df_epoch[mask_degraded][TARGET_COL]
    y_pred_d = df_epoch[mask_degraded]["Predicted_Energy_kW"]
    r2_degraded = r2_score(y_true_d, y_pred_d) if not y_true_d.empty else np.nan
    
    logger.info("  Healthy Fleet R^2  : %.4f (Model retains accuracy)", r2_healthy)
    logger.info("  Degraded Fleet R^2 : %.4f (Metric collapse confirms anomaly)", r2_degraded)
    logger.info(separator)

    anomaly_rates: Dict[str, float] = {}
    for buoy in BUOY_ORDER:
        df_buoy: pd.DataFrame = df_epoch[df_epoch[BUOY_ID_COL] == buoy]
        if df_buoy.empty:
            continue
        pct: float = 100.0 * int(df_buoy["Is_Absolute_Anomaly"].sum()) / len(df_buoy)
        anomaly_rates[buoy] = pct
        logger.info("%s underperformed in %.2f%% of the timestamps.", buoy.replace("_", " "), pct)

    if anomaly_rates:
        worst_buoy: str = max(anomaly_rates, key=lambda b: anomaly_rates[b])
        logger.info(separator)
        logger.info("CONCLUSION -- Worst performing asset: %s (anomaly rate = %.2f%%).", worst_buoy.replace("_", " "), anomaly_rates[worst_buoy])

def export_artefacts(df: pd.DataFrame, model: XGBRegressor, rmse_test: float) -> None:
    logger.info("Stage H -- Exporting intermediate artefacts")
    export_cols = [TIMESTAMP_COL, BUOY_ID_COL, "Predicted_Energy_kW", "Absolute_Residual", "Is_Absolute_Anomaly"]
    df_export = df[export_cols].copy()
    df_export["RMSE_test_dynamic"] = rmse_test
    
    os.makedirs(os.path.dirname(PHASE1_CSV_OUT), exist_ok=True)
    df_export.to_csv(PHASE1_CSV_OUT, index=False)
    joblib.dump(model, PHASE1_MODEL_OUT)
    logger.info("Export complete.")

## Execution Block
Chamada sequencial das funções para correr a Fase 1 na totalidade (equivalente ao antigo bloco `__main__`).

In [7]:
logger.info("=" * 60)
logger.info("WEC Phase 1 -- Absolute Performance Analysis Execution")
logger.info("=" * 60)

# Stage A
df = load_and_engineer_features(DATA_PATH)

# Stage B
X_train, y_train, X_test, y_test, buoys_train, buoys_test = temporal_split(df)

# Stage C
model, rmse_test = train_and_evaluate(X_train, y_train, X_test, y_test, buoys_test)

# Stages D + E
df = run_inference_and_flag(df, model, rmse_test)

# Stage F
generate_figure(df, model, rmse_test)

# Stage G
emit_asset_performance_report(df, epoch=3)

# Stage H 
export_artefacts(df, model, rmse_test)

logger.info("=" * 60)
logger.info("Pipeline completed successfully.")
logger.info("=" * 60)


2026-06-09 15:36:18 | INFO | ============================================================
2026-06-09 15:36:18 | INFO | WEC Phase 1 -- Absolute Performance Analysis Execution
2026-06-09 15:36:18 | INFO | ============================================================
2026-06-09 15:36:18 | INFO | Stage A -- Loading data from: dataset2/wec_c5_mock_data_epochs.csv
2026-06-09 15:36:19 | INFO | Raw shape: (86976, 17)
2026-06-09 15:36:19 | INFO | Imputing 777 missing value(s) via median strategy
2026-06-09 15:36:19 | INFO | Engineered feature 'Wave_Power_Flux' computed (0.49 * Hs^2 * Te)
2026-06-09 15:36:19 | INFO | Temporal features extracted: hour, month
2026-06-09 15:36:19 | INFO | Preprocessed shape: (86976, 19)
2026-06-09 15:36:19 | INFO | Stage B -- Temporal split: train=69580 rows | test=17396 rows (80% / 20%)
2026-06-09 15:36:19 | INFO | Stage C -- Training XGBoost on 69580 samples, 15 features...
2026-06-09 15:36:20 | INFO | Training complete.
2026-06-09 15:36:20 | INFO | --- Baseline M